In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras import layers, models, callbacks

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
LOOKBACK = 24
HORIZON  = 1

In [3]:
df = pd.read_csv("../data/LSTM-Multivariate_pollution.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("h")

In [4]:
n_zero = (df["pollution"] == 0).sum()
print(f"pollution == 0 olan satır: {n_zero}")
df["pollution"] = df["pollution"].replace(0, np.nan).interpolate(limit=24)
df = df.dropna()

pollution == 0 olan satır: 2045


In [5]:
df = pd.get_dummies(df, columns=["wnd_dir"], dtype=float)

In [6]:
df["pollution"] = np.log1p(df["pollution"])
df = df[["pollution"] + [c for c in df.columns if c != "pollution"]]

In [7]:
train = df.loc[:"2014-06-30"]
val   = df.loc["2014-07-01":"2014-11-30"]
test  = df.loc["2014-12-01":]
print("train:", train.shape, "val:", val.shape, "test:", test.shape)

train: (38523, 11) val: (3672, 11) test: (744, 11)


In [8]:
scaler = MinMaxScaler().fit(train)

In [9]:
def to_scaled(part):
    return scaler.transform(part)

In [10]:
def make_windows(data, lookback=LOOKBACK, horizon=HORIZON):
    X, y = [], []
    for i in range(lookback, len(data) - horizon + 1):
        X.append(data[i - lookback:i])
        y.append(data[i + horizon - 1, 0])   
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [11]:
Xtr, ytr = make_windows(to_scaled(train))
Xva, yva = make_windows(to_scaled(pd.concat([train.tail(LOOKBACK), val])))
Xte, yte = make_windows(to_scaled(pd.concat([val.tail(LOOKBACK), test])))
print("Xtr:", Xtr.shape, "Xva:", Xva.shape, "Xte:", Xte.shape)

Xtr: (38499, 24, 11) Xva: (3672, 24, 11) Xte: (744, 24, 11)


In [12]:
n_features = Xtr.shape[2]

In [13]:
model = models.Sequential([
    layers.Input(shape=(LOOKBACK, n_features)),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),
])

In [14]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="huber")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          19,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

In [16]:
history = model.fit(
    Xtr, ytr,
    validation_data=(Xva, yva),
    epochs=100,
    batch_size=128,
    callbacks=cb,
    verbose=1,
)

Epoch 1/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 26s 60ms/step - loss: 0.0052 - val_loss: 0.0018 - learning_rate: 0.0010
Epoch 2/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 18s 59ms/step - loss: 0.0022 - val_loss: 0.0014 - learning_rate: 0.0010
Epoch 3/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 18s 59ms/step - loss: 0.0016 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 4/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 18s 60ms/step - loss: 0.0013 - val_loss: 0.0012 - learning_rate: 0.0010
Epoch 5/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 20s 58ms/step - loss: 0.0012 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 6/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 21s 58ms/step - loss: 0.0011 - val_loss: 9.6870e-04 - learning_rate: 0.0010
Epoch 7/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 17s 58ms/step - loss: 0.0011 - val_loss: 9.1500e-04 - learning_rate: 0.0010
Epoch 8/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 18s 58ms/step - loss: 0.0010 - val_loss: 9.7630e-04 - learning_rate: 0.0010
Epoch 9/100
301/301 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - loss: 0.0010 - val_loss: 0.

In [17]:
def inverse_pollution(y_scaled):
    # MinMaxScaler: x_scaled = x * scale_ + min_  →  x = (x_scaled - min_) / scale_
    y_log = (y_scaled - scaler.min_[0]) / scaler.scale_[0]
    return np.expm1(y_log)

In [18]:
pred_te = inverse_pollution(model.predict(Xte, verbose=0).ravel())
true_te = inverse_pollution(yte)

In [19]:
prev_te = inverse_pollution(Xte[:, -1, 0])

In [20]:
def report(name, y, p):
    mae  = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    print(f"{name:<12} MAE={mae:7.2f}   RMSE={rmse:7.2f}")

In [21]:
report("Persistence", true_te, prev_te)
report("LSTM", true_te, pred_te)

Persistence  MAE=  12.59   RMSE=  24.47
LSTM         MAE=  13.46   RMSE=  24.10


In [22]:
model.save("../model/air_pollution_lstm.keras")